# 🔍 Phase 2 — Exploratory Data Analysis (EDA)

**Author:** Saed Abdalgani

> **Malicious PDF Detector** | Structural Feature Analysis  
> Objective: Deep-dive into the CIC PDFMal2022 dataset to uncover statistical signatures, identify top discriminative features, and validate data quality ahead of model training.

---

In [ ]:
# ── Cell 1: Setup & Load Data ─────────────────────────────────────────────
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

# Ensure project root is importable
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import (
    PROCESSED_DATA_DIR, RAW_DATA_DIR, FEATURE_COLUMNS,
    FIGURES_DIR, RESULTS_DIR
)
from src.data.loader import load_dataset, get_feature_matrix
from src.utils.visualization import (
    plot_class_distribution, plot_feature_distributions,
    plot_correlation_matrix, plot_boxplots, plot_pairplot,
    plot_pvalue_heatmap, plot_outlier_summary,
    plot_feature_importance, COLORS, _apply_dark_theme
)

%matplotlib inline
plt.rcParams['figure.max_open_warning'] = 0

# ── Load the datasets ──
print("="*60)
print("  📂 LOADING DATASETS")
print("="*60)

# Load raw dataset for full analysis
raw_path = RAW_DATA_DIR / "pdfmal2022.csv"
if raw_path.exists():
    df_raw = pd.read_csv(raw_path)
    print(f"✅ Raw dataset loaded: {df_raw.shape}")
else:
    print("⚠️  Raw dataset not found. Run Phase 1 downloader first.")
    df_raw = None

# Load cleaned dataset
cleaned_path = PROCESSED_DATA_DIR / "cleaned.csv"
if cleaned_path.exists():
    df_cleaned = pd.read_csv(cleaned_path)
    print(f"✅ Cleaned dataset loaded: {df_cleaned.shape}")
else:
    print("⚠️  Cleaned dataset not found. Using raw data instead.")
    df_cleaned = df_raw.copy() if df_raw is not None else None

# Load train/val/test splits
train_path = PROCESSED_DATA_DIR / "train.csv"
val_path = PROCESSED_DATA_DIR / "val.csv"
test_path = PROCESSED_DATA_DIR / "test.csv"

splits_exist = all(p.exists() for p in [train_path, val_path, test_path])
if splits_exist:
    df_train = pd.read_csv(train_path)
    df_val = pd.read_csv(val_path)
    df_test = pd.read_csv(test_path)
    print(f"✅ Train split loaded: {df_train.shape}")
    print(f"✅ Val split loaded:   {df_val.shape}")
    print(f"✅ Test split loaded:  {df_test.shape}")
else:
    print("⚠️  Split files not found. Some analyses will run on the cleaned set.")
    df_train = df_val = df_test = None

# Use the best available dataset for EDA
df = df_cleaned if df_cleaned is not None else df_raw
label_col = 'Class' if 'Class' in df.columns else df.columns[-1]
feature_cols = [c for c in FEATURE_COLUMNS if c in df.columns]

print(f"\n📊 Working dataset shape: {df.shape}")
print(f"🏷️  Label column: '{label_col}'")
print(f"🔢 Feature columns: {len(feature_cols)}")
print(f"\n{df.head(3)}")

---
## 📊 Cell 2 — Class Distribution Analysis
Understanding the class balance is critical before training. Imbalanced datasets require special handling (SMOTE, class weights).

In [ ]:
# ── Cell 2: Class Distribution ────────────────────────────────────────────
print("="*60)
print("  📊 CLASS DISTRIBUTION ANALYSIS")
print("="*60)

# Numeric breakdown
class_counts = df[label_col].value_counts()
class_pcts = df[label_col].value_counts(normalize=True) * 100

print("\n🔢 Class Counts:")
for cls, count in class_counts.items():
    pct = class_pcts[cls]
    emoji = "🟢" if cls == 0 else "🔴"
    label = "Benign" if cls == 0 else "Malicious"
    print(f"  {emoji} {label} (class {cls}): {count:>6,} samples ({pct:.2f}%)")

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\n⚖️  Imbalance Ratio: {imbalance_ratio:.2f}:1")

if imbalance_ratio > 2:
    print("   ⚠️  Significant imbalance detected — SMOTE is justified.")
else:
    print("   ✅ Relatively balanced dataset.")

# Visualization
fig = plot_class_distribution(
    df, save_path=str(FIGURES_DIR / "class_distribution.png"),
    label_col=label_col
)
plt.show()

# Split-level distribution comparison
if splits_exist:
    print("\n📋 Class Ratios Across Splits:")
    split_stats = []
    for name, sdf in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
        vc = sdf[label_col].value_counts(normalize=True) * 100
        stats_row = {"Split": name, "Total": len(sdf)}
        for cls in sorted(vc.index):
            lbl = "Benign %" if cls == 0 else "Malicious %"
            stats_row[lbl] = f"{vc[cls]:.2f}%"
        split_stats.append(stats_row)
    display(pd.DataFrame(split_stats).set_index("Split"))

---
## 📈 Cell 3 — Univariate Analysis
Overlaid histograms for all 37 features, revealing how each feature's distribution differs between malicious and benign PDFs.

In [ ]:
# ── Cell 3: Univariate Feature Distributions ──────────────────────────────
print("="*60)
print("  📈 UNIVARIATE FEATURE DISTRIBUTIONS")
print("="*60)

# Basic statistics per class
print("\n📊 Descriptive Statistics (Overall):")
display(df[feature_cols].describe().round(3).T.style.background_gradient(
    cmap='YlOrRd', subset=['mean', 'std', 'max']
))

# Per-class means comparison
print("\n🔍 Mean Values by Class:")
class_means = df.groupby(label_col)[feature_cols].mean().T
class_means.columns = ['Benign (Mean)' if c == 0 else 'Malicious (Mean)' for c in class_means.columns]
class_means['Difference'] = class_means.iloc[:, 1] - class_means.iloc[:, 0]
class_means['Abs Diff'] = class_means['Difference'].abs()
class_means = class_means.sort_values('Abs Diff', ascending=False)
display(class_means.round(4).style.background_gradient(
    cmap='RdYlGn_r', subset=['Abs Diff']
))

# Identify top-10 most visually discriminative features
top10_by_diff = class_means.head(10).index.tolist()
print(f"\n🏆 Top 10 Features by Mean Difference:")
for i, feat in enumerate(top10_by_diff, 1):
    diff = class_means.loc[feat, 'Abs Diff']
    print(f"  {i:>2}. {feat:<25} (Δ = {diff:.4f})")

# Full distribution plot
fig = plot_feature_distributions(
    df, features=feature_cols,
    save_path=str(FIGURES_DIR / "feature_distributions.png"),
    label_col=label_col
)
plt.show()

---
## 🧪 Cell 4 — Statistical Significance Testing
Mann-Whitney U test per feature — identifies which features have statistically significant differences between classes.

In [ ]:
# ── Cell 4: Mann-Whitney U Test ──────────────────────────────────────────
print("="*60)
print("  🧪 STATISTICAL SIGNIFICANCE TESTING")
print("="*60)

classes = sorted(df[label_col].unique())
group_a = df[df[label_col] == classes[0]]  # Benign
group_b = df[df[label_col] == classes[-1]]  # Malicious

significance_results = []
for feat in feature_cols:
    a = group_a[feat].dropna()
    b = group_b[feat].dropna()
    
    try:
        statistic, p_value = stats.mannwhitneyu(a, b, alternative='two-sided')
        # Effect size: rank-biserial correlation
        n1, n2 = len(a), len(b)
        effect_size = 1 - (2 * statistic) / (n1 * n2)
    except Exception:
        statistic, p_value, effect_size = np.nan, 1.0, 0.0
    
    significance_results.append({
        'feature': feat,
        'U_statistic': statistic,
        'p_value': p_value,
        'effect_size': abs(effect_size),
        'significant_001': '✅' if p_value < 0.001 else '❌',
        'significant_01': '✅' if p_value < 0.01 else '❌',
        'significant_05': '✅' if p_value < 0.05 else '❌',
    })

sig_df = pd.DataFrame(significance_results).sort_values('p_value')

# Summary
n_sig_001 = (sig_df['p_value'] < 0.001).sum()
n_sig_01 = (sig_df['p_value'] < 0.01).sum()
n_sig_05 = (sig_df['p_value'] < 0.05).sum()

print(f"\n📊 Significance Summary (out of {len(feature_cols)} features):")
print(f"  • p < 0.001 (highly significant): {n_sig_001} features")
print(f"  • p < 0.01  (very significant):   {n_sig_01} features")
print(f"  • p < 0.05  (significant):        {n_sig_05} features")
print(f"  • p ≥ 0.05  (not significant):    {len(feature_cols) - n_sig_05} features")

# Display sorted table
print("\n📋 Full Significance Table (sorted by p-value):")
display(
    sig_df[['feature', 'p_value', 'effect_size', 'significant_001', 'significant_01']]
    .reset_index(drop=True)
    .style.format({'p_value': '{:.2e}', 'effect_size': '{:.4f}'})
    .background_gradient(cmap='YlOrRd_r', subset=['p_value'])
    .background_gradient(cmap='YlOrRd', subset=['effect_size'])
)

# Save significance results
sig_df.to_csv(RESULTS_DIR / 'feature_significance.csv', index=False)
print(f"\n💾 Saved to {RESULTS_DIR / 'feature_significance.csv'}")

# Plot p-value heatmap
fig = plot_pvalue_heatmap(
    sig_df[['feature', 'p_value']],
    save_path=str(FIGURES_DIR / "pvalue_significance.png")
)
plt.show()

---
## 🔗 Cell 5 — Correlation Analysis
Full correlation matrix to identify highly correlated feature pairs (|r| > 0.9), which may indicate redundancy.

In [ ]:
# ── Cell 5: Correlation Analysis ─────────────────────────────────────────
print("="*60)
print("  🔗 CORRELATION ANALYSIS")
print("="*60)

corr_matrix = df[feature_cols].corr()

# Find highly correlated pairs (|r| > 0.9)
high_corr_pairs = []
for i in range(len(feature_cols)):
    for j in range(i+1, len(feature_cols)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.9:
            high_corr_pairs.append({
                'Feature 1': feature_cols[i],
                'Feature 2': feature_cols[j],
                'Pearson r': r,
                '|r|': abs(r),
            })

if high_corr_pairs:
    hc_df = pd.DataFrame(high_corr_pairs).sort_values('|r|', ascending=False)
    print(f"\n⚠️  Found {len(hc_df)} highly correlated pairs (|r| > 0.9):")
    display(
        hc_df.style.format({'Pearson r': '{:.4f}', '|r|': '{:.4f}'})
        .background_gradient(cmap='Reds', subset=['|r|'])
    )
    print("\n💡 Discussion: Highly correlated features may be candidates for removal")
    print("   to reduce multicollinearity and model complexity. However, tree-based")
    print("   models are generally robust to correlated features. Consider removing")
    print("   only if using linear models or if feature count needs reduction.")
else:
    print("\n✅ No highly correlated pairs found (|r| > 0.9).")

# Moderately correlated pairs (0.7 < |r| <= 0.9)
moderate_corr = []
for i in range(len(feature_cols)):
    for j in range(i+1, len(feature_cols)):
        r = corr_matrix.iloc[i, j]
        if 0.7 < abs(r) <= 0.9:
            moderate_corr.append({
                'Feature 1': feature_cols[i],
                'Feature 2': feature_cols[j],
                'Pearson r': r,
            })

print(f"\n📊 Moderate correlations (0.7 < |r| ≤ 0.9): {len(moderate_corr)} pairs")

# Full correlation matrix heatmap
fig = plot_correlation_matrix(
    df, features=feature_cols,
    save_path=str(FIGURES_DIR / "correlation_matrix.png")
)
plt.show()

---
## 🎯 Cell 6 — Bivariate Analysis
Pair plots of the top-5 most discriminative features to visualize separability between classes.

In [ ]:
# ── Cell 6: Bivariate Analysis — Pair Plots ─────────────────────────────
print("="*60)
print("  🎯 BIVARIATE ANALYSIS")
print("="*60)

# Select top-5 features by significance (from Cell 4 results)
top5_features = sig_df.head(5)['feature'].tolist()
print(f"\n🏆 Top 5 Features for Pair Plot:")
for i, feat in enumerate(top5_features, 1):
    p = sig_df[sig_df['feature'] == feat]['p_value'].values[0]
    eff = sig_df[sig_df['feature'] == feat]['effect_size'].values[0]
    print(f"  {i}. {feat:<25} p={p:.2e}, effect_size={eff:.4f}")

# Pair plot
fig = plot_pairplot(
    df, top_features=top5_features,
    save_path=str(FIGURES_DIR / "pairplot_top5.png"),
    label_col=label_col
)
plt.show()

# Additional: 2D scatter of the two most discriminative features
_apply_dark_theme()
if len(top5_features) >= 2:
    fig2, ax = plt.subplots(figsize=(10, 8))
    for cls in sorted(df[label_col].unique()):
        subset = df[df[label_col] == cls]
        color = COLORS['benign'] if cls == classes[0] else COLORS['malicious']
        label = 'Benign' if cls == classes[0] else 'Malicious'
        ax.scatter(
            subset[top5_features[0]], subset[top5_features[1]],
            c=color, label=label, alpha=0.5, s=20, edgecolors='none'
        )
    ax.set_xlabel(top5_features[0], fontsize=12)
    ax.set_ylabel(top5_features[1], fontsize=12)
    ax.set_title(f'2D Scatter: {top5_features[0]} vs {top5_features[1]}', fontsize=14, pad=15)
    ax.legend(fontsize=11)
    fig2.savefig(FIGURES_DIR / 'scatter_top2.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 📦 Cell 7 — Outlier Analysis
Box plots for the top-10 features by class, with IQR outlier count tables.

In [ ]:
# ── Cell 7: Outlier Analysis ─────────────────────────────────────────────
print("="*60)
print("  📦 OUTLIER ANALYSIS")
print("="*60)

# Calculate IQR outlier counts per feature
outlier_counts = {}
outlier_details = []

for feat in feature_cols:
    Q1 = df[feat].quantile(0.25)
    Q3 = df[feat].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = ((df[feat] < lower) | (df[feat] > upper))
    n_outliers = outliers.sum()
    pct_outliers = n_outliers / len(df) * 100
    outlier_counts[feat] = n_outliers
    
    # Per-class outlier breakdown
    outlier_benign = ((df[label_col] == classes[0]) & outliers).sum()
    outlier_malicious = ((df[label_col] == classes[-1]) & outliers).sum()
    
    outlier_details.append({
        'Feature': feat,
        'Total Outliers': n_outliers,
        '% of Data': f"{pct_outliers:.2f}%",
        'Benign Outliers': outlier_benign,
        'Malicious Outliers': outlier_malicious,
        'IQR': f"{IQR:.4f}",
        'Lower Bound': f"{lower:.4f}",
        'Upper Bound': f"{upper:.4f}",
    })

outlier_df = pd.DataFrame(outlier_details).sort_values('Total Outliers', ascending=False)

print("\n📋 IQR Outlier Count Table (sorted by count):")
display(
    outlier_df.head(15).reset_index(drop=True)
    .style.background_gradient(cmap='YlOrRd', subset=['Total Outliers'])
)

total_outlier_cells = sum(outlier_counts.values())
print(f"\n📊 Total outlier cells: {total_outlier_cells:,}")
print(f"   Features with outliers: {sum(1 for v in outlier_counts.values() if v > 0)} / {len(feature_cols)}")

# Save outlier analysis
outlier_df.to_csv(RESULTS_DIR / 'outlier_analysis.csv', index=False)
print(f"💾 Saved to {RESULTS_DIR / 'outlier_analysis.csv'}")

# Box plots for top-10 most outlier-heavy features
top10_outlier_features = outlier_df.head(10)['Feature'].tolist()

fig = plot_boxplots(
    df, features=top10_outlier_features,
    save_path=str(FIGURES_DIR / "boxplots_top10.png"),
    label_col=label_col,
    title="Box Plots — Top 10 Features by Outlier Count"
)
plt.show()

# Outlier summary bar chart
fig2 = plot_outlier_summary(
    pd.Series(outlier_counts),
    save_path=str(FIGURES_DIR / "outlier_summary.png")
)
plt.show()

---
## 🏁 Cell 8 — Key Findings Summary
Consolidation of all EDA findings with actionable insights for the feature engineering and model training phases.

In [ ]:
# ── Cell 8: Key Findings Summary ─────────────────────────────────────────
print("="*70)
print("  🏁 KEY FINDINGS SUMMARY — EXPLORATORY DATA ANALYSIS")
print("="*70)

# ── 1. Dataset Overview ──
print("\n📊 1. DATASET OVERVIEW")
print(f"   • Total samples: {len(df):,}")
print(f"   • Total features: {len(feature_cols)}")
print(f"   • Classes: {df[label_col].nunique()} (Benign / Malicious)")
print(f"   • Imbalance ratio: {imbalance_ratio:.2f}:1")

# ── 2. Top Discriminative Features ──
print("\n🏆 2. TOP 10 DISCRIMINATIVE FEATURES")
print("   (Ranked by statistical significance and effect size)")

top10_summary = []
for i, (_, row) in enumerate(sig_df.head(10).iterrows(), 1):
    feat = row['feature']
    p = row['p_value']
    eff = row['effect_size']
    
    # Determine rationale
    benign_mean = group_a[feat].mean()
    mal_mean = group_b[feat].mean()
    direction = "higher" if mal_mean > benign_mean else "lower"
    
    rationale = f"Malicious PDFs have {direction} {feat}"
    if 'js' in feat.lower() or 'javascript' in feat.lower():
        rationale += " (JavaScript execution indicator)"
    elif 'action' in feat.lower() or 'openaction' in feat.lower():
        rationale += " (auto-execution trigger)"
    elif 'obj' in feat.lower():
        rationale += " (structural complexity marker)"
    elif 'stream' in feat.lower():
        rationale += " (embedded content indicator)"
    elif 'obfuscation' in feat.lower():
        rationale += " (evasion technique detector)"
    elif 'uri' in feat.lower():
        rationale += " (external resource reference)"
    
    top10_summary.append({
        'Rank': i,
        'Feature': feat,
        'p-value': f"{p:.2e}",
        'Effect Size': f"{eff:.4f}",
        'Benign Mean': f"{benign_mean:.3f}",
        'Malicious Mean': f"{mal_mean:.3f}",
        'Rationale': rationale,
    })

summary_df = pd.DataFrame(top10_summary)
display(summary_df.set_index('Rank'))

# ── 3. Correlation Findings ──
print("\n🔗 3. CORRELATION FINDINGS")
print(f"   • Highly correlated pairs (|r| > 0.9): {len(high_corr_pairs)}")
print(f"   • Moderately correlated pairs (0.7 < |r| ≤ 0.9): {len(moderate_corr)}")
if high_corr_pairs:
    print("   • Recommendation: Monitor during model training; tree-based models")
    print("     handle correlated features well. Consider removal for MLP.")

# ── 4. Outlier Findings ──
print("\n📦 4. OUTLIER FINDINGS")
print(f"   • Total outlier cells (IQR method): {total_outlier_cells:,}")
print(f"   • Features with outliers: {sum(1 for v in outlier_counts.values() if v > 0)} / {len(feature_cols)}")
print("   • Strategy: Outliers flagged but NOT removed — they may be")
print("     legitimate indicators of malicious PDF characteristics.")

# ── 5. Conclusions ──
print("\n✅ 5. CONCLUSIONS FOR NEXT PHASES")
print("   📌 Feature Engineering:")
print(f"      - All {n_sig_001} highly significant features should be retained.")
print(f"      - Consider feature interactions for top-5 discriminators.")
print("   📌 Model Training:")
print("      - SMOTE is justified given the imbalance ratio.")
print("      - Tree-based models (RF, XGB, LGBM) are well-suited for this data.")
print("      - MLP may benefit from feature standardization.")
print("   📌 Security Insight:")
print("      - JavaScript and action-related features are top discriminators,")
print("        aligning with known PDF exploitation vectors.")

# ── Save summary to CSV ──
summary_df.to_csv(RESULTS_DIR / 'eda_top10_features.csv', index=False)
print(f"\n💾 Summary saved to {RESULTS_DIR / 'eda_top10_features.csv'}")

print("\n" + "="*70)
print("  🎉 EDA COMPLETE — Ready for Phase 3: Feature Engineering!")
print("="*70)